<a href="https://colab.research.google.com/github/pandeyayush260804/FIFA_MATCHES_DV/blob/main/Copy_of_Student_Exercises_RAG_Vector_Databases_(14).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Exercises: RAG & Vector Databases

**Post-class practice!** In this notebook, you will build your own mini RAG system step by step.

**Instructions:**
- Complete the lines marked with `# TODO` in each exercise
- Run the code and observe the output carefully
- Answer every "Reflection Question" in the markdown cell provided

**Setup:** Run this on Google Colab (Runtime → Change runtime type → GPU recommended, but CPU works too — just slower)


---
## Part 0: Setup

First, install and import the required libraries.


In [ ]:
# Run this first!
!pip install transformers torch sentence-transformers faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 67.1 MB/s eta 0:00:00


In [ ]:
from transformers import pipeline

# Load the text generation model (same as class)
generator = pipeline('text-generation', model='gpt2')

print("Setup complete! ✅")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Setup complete! ✅


---
## Part 1: Hallucination vs RAG — The "Open Book Exam" 📖

Remember: without context, the LLM **guesses** (Hallucination). With RAG, we give it the "book" to read from.


### Exercise 1.1: Catch the Hallucination! 🕵️

**Task:** Ask GPT-2 a question about something it **cannot possibly know** (a made-up or very recent fact). Observe how it confidently makes something up.

Example ideas: *"Who won the 2027 Cricket World Cup?"*, *"What is the name of the mayor of Atlantis?"*


In [ ]:
# TODO: Write a question the model cannot know the answer to
my_question = "Who will win the 2026 Formula 1 Abu Dhabi Grand Prix?"  # <-- your impossible question here

res = generator(my_question, max_new_tokens=30)
print(res[0]['generated_text'])

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Who will win the 2026 Formula 1 Abu Dhabi Grand Prix?

If you're still not convinced, take a look at the official standings and follow the links below to get a better idea of who you can


**🤔 Reflection Question 1.1:** Did the model say "I don't know"? Or did it confidently invent an answer? Why do you think LLMs hallucinate instead of admitting they don't know?

> *The model did not say "I don't know." Instead, it generated a likely but invented answer. LLMs hallucinate because they generate the most probable sequence of words based on patterns learned during training rather than verifying facts or knowing when information is unavailable.*


### Exercise 1.2: Fix It with RAG (Manual Augmentation) 🔧

**Task:** Now fix the hallucination from Exercise 1.1 using the **Augmentation** step:
1. Write a `private_knowledge` string that contains the correct answer to your question
2. Build a `rag_prompt` that combines **Context + Question** (like we did in class)
3. Compare the output with Exercise 1.1


In [ ]:
# TODO: Write the "ground truth" for your question
private_knowledge = "The 2026 Formula 1 Abu Dhabi Grand Prix was won by Oscar Piastri."

# TODO: Build the RAG prompt — attach context to the question
# Format: Context: ... \n\n Question: ... \n Answer according to the context:
rag_prompt = f"""
Context:
{private_knowledge}

Question:
{my_question}

Answer according to the context:
"""   # <-- build your prompt here using private_knowledge and my_question

res_rag = generator(rag_prompt, max_new_tokens=30)
print(res_rag[0]['generated_text'])

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Context:
The 2026 Formula 1 Abu Dhabi Grand Prix was won by Oscar Piastri.

Question:
Who will win the 2026 Formula 1 Abu Dhabi Grand Prix?

Answer according to the context:

Andretti

Estevez

Pasquale

Cabrera

Rams

Hangzhou




**🤔 Reflection Question 1.2:** Which of the 3 RAG steps (Retrieval, Augmentation, Generation) did we do **manually** here? Which step is still missing from our system?

> *We manually performed the Augmentation step by adding the correct context to the prompt. The Retrieval step is still missing because the context was manually written instead of being retrieved automatically from a vector database.*


---
## Part 2: Embeddings — Turning Meaning into Numbers 🔢

Remember: AI understands **Numbers**, not words. Similar meanings → vectors that are **close** on the graph.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the embedding model (same as class)
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model ready! ✅")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model ready! ✅


### Exercise 2.1: Look Inside an Embedding 👀

**Task:** Convert a sentence into an embedding and inspect it:
1. Encode the sentence `"I love programming"`
2. Print the **shape** (how many numbers?) and the **first 10 numbers**


In [ ]:
sentence = "I love programming"

# TODO: encode the sentence using model.encode()
embedding = model.encode(sentence)   # <-- fix this

print("Shape of embedding:", embedding.shape)
print("First 10 numbers:", embedding[:10])

Shape of embedding: (384,)
First 10 numbers: [-0.03617864 -0.0127737   0.00300628 -0.01690346  0.00948426 -0.06515172
  0.09376637  0.07142349  0.01852629  0.05358268]


**🤔 Reflection Question 2.1:** How many dimensions (numbers) does one embedding have? Can a human read meaning from these numbers directly?

> *The embedding has 384 dimensions. These numbers represent the semantic meaning of the sentence, but humans cannot directly understand the meaning by looking at the numbers.*


### Exercise 2.2: The King & Queen Test 👑

In class we said: *"King" and "Queen" vectors will be close, but "Apple" will be far away.* Let's **prove it with code**!

**Task:** Compute the similarity between sentence pairs. Higher score = closer meaning.


In [ ]:
from sentence_transformers import util

sentences = [
    "The king rules the country.",      # 0
    "The queen lives in the palace.",   # 1
    "I ate an apple for breakfast."     # 2
]

embeddings = model.encode(sentences)

# TODO: compute cosine similarity between sentence 0 and sentence 1 (king vs queen)
sim_king_queen = util.cos_sim(embeddings[0], embeddings[1])   # <-- fix the index

# TODO: compute cosine similarity between sentence 0 and sentence 2 (king vs apple)
sim_king_apple = util.cos_sim(embeddings[0], embeddings[2])   # <-- fix the index

print(f"King vs Queen similarity: {sim_king_queen.item():.4f}")
print(f"King vs Apple similarity: {sim_king_apple.item():.4f}")

King vs Queen similarity: 0.4106
King vs Apple similarity: 0.0614


**🤔 Reflection Question 2.2:** Which pair got the higher similarity score? Does this match the "close on the graph" logic from class?

> *King and queen has high similarity score...*


---
## Part 3: Build Your Own Vector Database 🗄️

Now the real deal — build a FAISS vector database with **your own documents** and search it.


### Exercise 3.1: Create Your Knowledge Base

**Task:** Write **5 documents of your own** (facts about your college, your city, your favorite topics — anything!). Then:
1. Encode them into vectors
2. Store them in a FAISS index
3. Print how many documents got stored


In [ ]:
import faiss

# TODO: Write your own 5 documents (make them about different topics!)
documents = [
    "Formula 1 has 10 teams and each team races with two drivers.",
    "DRS (Drag Reduction System) helps Formula 1 drivers overtake another car by opening the rear wing on designated zones, reducing drag and increasing speed.",
    "Soft tyres provide the highest grip but wear out quickly. Hard tyres last longer but provide less grip.",
    "Monaco Grand Prix is one of the most famous races in Formula 1 and is held on a street circuit.",
    "Pit stops are used to change tyres, repair damage, or adjust race strategy during a Grand Prix."
]

# TODO: Convert documents into vectors using model.encode()
doc_embeddings = np.array(model.encode(documents), dtype="float32")

dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

print(f"{index.ntotal} documents have been stored in the Vector Database.")

5 documents have been stored in the Vector Database.


### Exercise 3.2: Semantic Search Test 🔍

**Task:** Ask a question about one of YOUR documents — but **do NOT use the exact same words** as the document! This tests **Semantic Search** (meaning) vs **Keyword Search** (exact words).

Example: if your document says *"Almonds are healthy for the brain"*, ask *"Which food is good for memory?"*


In [ ]:
# TODO: Write a query that matches one document by MEANING, not exact words
query = "What is DRS in Formula 1?"  # <-- your query

# TODO: Convert the query into a vector
query_embedding = np.array(model.encode([query]), dtype="float32")  # <-- fix (hint: model.encode([query]))

# TODO: Search the database for the top 1 result
D, I = index.search(np.array(query_embedding), k=1)   # <-- fix k

print(f"Question: {query}")
print(f"Most similar Document: {documents[I[0][0]]}")
print(f"Distance: {D[0][0]:.4f}  (lower = more similar)")

Question: What is DRS in Formula 1?
Most similar Document: DRS (Drag Reduction System) helps Formula 1 drivers overtake another car by opening the rear wing on designated zones, reducing drag and increasing speed.
Distance: 0.5635  (lower = more similar)


**🤔 Reflection Question 3.2:** Did the database find the right document even though you used different words? Explain how this is different from Ctrl+F / keyword search:

> *Yes, the vector database retrieved the document that was most semantically related to my query. Unlike keyword search, semantic search compares the meaning of the query with the stored documents instead of matching exact words, allowing it to find relevant information even when different wording is used.*


### Exercise 3.3: Top-K Retrieval

**Task:** Instead of only the top 1 result, retrieve the **top 3** most similar documents and print them in order with their distances.


In [ ]:
query2 = "Which Formula 1 tyre provides the highest grip?"  # <-- write another query

query_embedding2 = model.encode([query2])

# TODO: search with k=3
D, I = index.search(query_embedding2, k=3)   # <-- fix k

print(f"Question: {query2}\n")
# TODO: loop over the 3 results and print rank, document, and distance
for rank in range(3):   # <-- fix range
    print(f"Rank {rank+1}: {documents[I[0][rank]]}  (distance: {D[0][rank]:.4f})")

Question: Which Formula 1 tyre provides the highest grip?

Rank 1: Soft tyres provide the highest grip but wear out quickly. Hard tyres last longer but provide less grip.  (distance: 0.6525)
Rank 2: Formula 1 has 10 teams and each team races with two drivers.  (distance: 1.2026)
Rank 3: Monaco Grand Prix is one of the most famous races in Formula 1 and is held on a street circuit.  (distance: 1.2527)


**🤔 Reflection Question 3.3:** Look at the distances of ranks 1, 2, and 3. Is the rank-1 document clearly the best match, or are the distances close? When might retrieving more than 1 document be useful for RAG?

> *The rank-1 document had the smallest distance, making it the most relevant match for the query. The rank-2 and rank-3 documents had larger distances, meaning they were less similar. Retrieving more than one document is useful in RAG because the required information may be spread across multiple documents, allowing the language model to generate a more complete and accurate answer.*


---
## Part 4: 🏆 Final Challenge — Full RAG Pipeline (End-to-End)

Now connect **everything**: Retrieval (Vector DB) → Augmentation (prompt building) → Generation (LLM).

This is the complete flow from class, but built by YOU:

```
User Query → [Vector DB Search] → Best Document → [Attach to Prompt] → [GPT-2] → Answer
```


In [ ]:
def my_rag_pipeline(user_query):
    # STEP 1 — RETRIEVAL
    q_emb = np.array(model.encode([user_query]), dtype="float32")

    # Search the FAISS index for the most relevant document
    D, I = index.search(q_emb, k=1)
    retrieved_doc = documents[I[0][0]]

    # STEP 2 — AUGMENTATION
    rag_prompt = f"""
Context:
{retrieved_doc}

Question:
{user_query}

Answer according to the context:
"""

    # STEP 3 — GENERATION
    res = generator(rag_prompt, max_new_tokens=50)

    print("Retrieved Document:")
    print(retrieved_doc)
    print("-" * 60)
    print("Final Answer:\n")
    print(res[0]["generated_text"])


# ===========================
# Test the RAG Pipeline
# ===========================

my_rag_pipeline("What is DRS in Formula 1?")

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieved Document:
DRS (Drag Reduction System) helps Formula 1 drivers overtake another car by opening the rear wing on designated zones, reducing drag and increasing speed.
------------------------------------------------------------
Final Answer:


Context:
DRS (Drag Reduction System) helps Formula 1 drivers overtake another car by opening the rear wing on designated zones, reducing drag and increasing speed.

Question:
What is DRS in Formula 1?

Answer according to the context:

DRS (Drag Reduction System) helps Formula 1 drivers overtake another car by opening the rear wing on designated zones, reducing drag and increasing speed.

DRS (Drag Reduction System) helps Formula 1 drivers overtake another car by opening the rear


**🤔 Final Reflection Question:**
1. In your pipeline, what would happen if the Vector DB retrieved the **wrong** document? Would the LLM still give a good answer?
2. Based on this, complete the sentence: *"A RAG system is only as good as its ______."*

> *If the Vector Database retrieves the wrong document, the language model will generate an answer based on incorrect or irrelevant context. Therefore, the final answer may also be inaccurate even though the language model is working correctly.*


### Bonus Challenge 4.1: RAG vs Fine-tuning Decision Table 🎓

For each scenario below, decide: **RAG or Fine-tuning?** (Remember: Fine-tuning = 5 years of med school, RAG = textbook in the exam)

| Scenario | RAG / Fine-tuning? | Reason |
|----------|-------------------|--------|
| A company chatbot that must answer from HR policy PDFs that change every month | ? | ? |
| Teaching a model to always respond in Shakespearean English style | ? | ? |
| A news assistant that must know today's headlines | ? | ? |
| A medical model that must deeply understand doctor-style reasoning | ? | ? |

> *Complete the table (edit this cell)...*


### Bonus Challenge 4.2: Break the Retriever! 💥

**Task:** Try to find a query where your Vector DB retrieves the **WRONG** document.

Hints to try:
- A very vague query (e.g., "Tell me something")
- A query about a topic NOT in any of your 5 documents
- A query mixing two topics at once

Print the result and the distance score.


In [ ]:
# TODO: try to fool your own retriever!
tricky_query = "can u tell why drs is wasteful"   # <-- your tricky query

q_emb = model.encode([tricky_query])
D, I = index.search(np.array(q_emb), k=1)

print(f"Question: {tricky_query}")
print(f"Retrieved: {documents[I[0][0]]}")
print(f"Distance: {D[0][0]:.4f}")

Question: can u tell why drs is wasteful
Retrieved: DRS (Drag Reduction System) helps Formula 1 drivers overtake another car by opening the rear wing on designated zones, reducing drag and increasing speed.
Distance: 1.2090


**🤔 Reflection Question 4.2:** What happened when you asked about a topic that was NOT in your database? Did the retriever say "not found", or did it still return *something*? Why is this a problem for real RAG systems, and how might we solve it? (Hint: think about the distance score!)

> *Your answer here...*


---
## ✅ Submission Checklist

Before submitting, check:

- [ ] Part 1: Hallucination caught + fixed with manual RAG
- [ ] Part 2: Embedding inspected + King/Queen similarity test done
- [ ] Part 3: Your own 5-document Vector DB built + semantic search + top-3 retrieval
- [ ] Part 4: Full end-to-end RAG pipeline working
- [ ] Bonus 4.1 table filled in
- [ ] All "Reflection Questions" answered
- [ ] **All cells have been run** (outputs are visible)

**Well done! 🎉 You have built a complete RAG system from scratch!**
